<a href='https://colab.research.google.com/github/Emelecto/QuantLab/blob/main/web/content/cursos/ml/notebooks/c5_l5.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg'/></a>

# C5-L5 · MLP vs LSTM
Foto (MLP) contra película (ventana secuencial): ¿paga la complejidad?

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path
URL = 'https://raw.githubusercontent.com/Emelecto/QuantLab/main/web/content/cursos/ml/data/c5_l5.csv'
try:
    df = pd.read_csv(URL)
    print('Fuente: URL (Colab)')
except Exception as e:
    print('Sin red, uso fallback local:', e)
    for cand in [Path('../data/c5_l5.csv'), Path('data/c5_l5.csv'), Path('c5_l5.csv')]:
        if cand.exists():
            df = pd.read_csv(cand); break
    print('Fuente: local')
print(df.shape)
print(df.head(5).to_string(index=False))

In [ ]:
# Features foto (lags en t) + secuencias de w=10 cierres normalizados
df['ret'] = df['close'].pct_change()
for k in range(1, 6):
    df[f'lag_{k}'] = df['ret'].shift(k)
df['vol10'] = df['ret'].rolling(10).std()
data = df.dropna().reset_index(drop=True)
W = 10
seqs = np.stack([data['ret'].values[i-W:i] for i in range(W, len(data))])
y_all = (data['ret'].shift(-1).values[W:] > 0).astype(int)
Xfoto = data[[f'lag_{k}' for k in range(1, 6)] + ['vol10']].values[W:]
print('Xfoto', Xfoto.shape, 'seqs', seqs.shape, 'y', y_all.shape)
assert Xfoto.shape[0] == seqs.shape[0] == y_all.shape[0] and len(y_all) > 100

In [ ]:
# MLP-foto: vector de t. Secuencial: MLP sobre ventana aplanada (proxy LSTM sin torch)
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
split = int(len(y_all)*0.7)
sc = StandardScaler().fit(Xfoto[:split])
mlp = MLPRegressor(hidden_layer_sizes=(32, 16), max_iter=500, early_stopping=True, n_iter_no_change=20, random_state=42)
mlp.fit(sc.transform(Xfoto[:split]), y_all[:split])
try:
    import torch  # noqa
    print('torch disponible: en Colab puedes cambiar el proxy por nn.LSTM real')
except Exception:
    print('sin torch: el rival secuencial es MLP sobre ventana (proxy honesto de LSTM)')
seq_flat = seqs.reshape(len(seqs), -1)
sc2 = StandardScaler().fit(seq_flat[:split])
lstm_proxy = MLPRegressor(hidden_layer_sizes=(32, 16), max_iter=500, early_stopping=True, n_iter_no_change=20, random_state=7)
lstm_proxy.fit(sc2.transform(seq_flat[:split]), y_all[:split])
print('iteraciones MLP:', mlp.n_iter_, ' proxy:', lstm_proxy.n_iter_)

In [ ]:
# Walk-forward de 2 folds: hit-rate OOS de cada uno
from sklearn.metrics import accuracy_score
edges = [int(len(y_all)*0.5), int(len(y_all)*0.75), len(y_all)]
for nombre, modelo, Xs, s in [('MLP', mlp, Xfoto, sc), ('SEC', lstm_proxy, seq_flat, sc2)]:
    hs = []
    for a, b in zip(edges[:-1], edges[1:]):
        hs.append(accuracy_score(y_all[a:b], (modelo.predict(s.transform(Xs[a:b])) > 0.5).astype(int)))
    print(f'{nombre}: hit folds={[round(h,3) for h in hs]} media={np.mean(hs):.3f}')
    globals()['hit_' + nombre.lower()] = float(np.mean(hs))

In [ ]:
# Chequeo automatico L5
assert np.isfinite(hit_mlp) and np.isfinite(hit_sec)
assert 0.3 <= hit_mlp <= 0.7 and 0.3 <= hit_sec <= 0.7, 'hit-rate debe ser sensato, no magico'
assert mlp.n_iter_ > 0 and lstm_proxy.n_iter_ > 0
print(f'OK L5: MLP {hit_mlp:.3f} vs SEC {hit_sec:.3f} — la complejidad debe ganarse su lugar')